# 02_bronze_entsoe_prices — Couche Bronze des prix day-ahead

**Projet :** Swiss Hydro & Power Analytics
**Source :** fichiers XML du Volume `energy_raw.landing`, dossier `entsoe/prices`
**Destination :** table Delta `energy_bronze.entsoe_prices_raw`

Ce notebook charge les fichiers XML déposés par `01_ingest_entsoe_prices` dans une table Delta.

**Principe de la couche Bronze :** les données sont conservées **telles qu'elles sont arrivées**, sans interprétation.
Chaque fichier devient une ligne contenant son XML complet, enrichie de métadonnées d'ingestion (fichier d'origine, date de chargement).
Le décodage du XML (positions, prix, horodatages) est fait dans la couche Silver.

**Pourquoi ne pas décoder dès maintenant ?** Si une erreur est découverte dans le décodage, il peut être refait à partir de Bronze, sans rappeler l'API.

| Étape | Description |
|---|---|
| 1 | Lecture des fichiers XML du landing |
| 2 | Ajout des métadonnées d'ingestion |
| 3 | Écriture dans la table Delta Bronze |


## 1. Lecture des fichiers du landing

Les fichiers sont lus avec `spark.read.text` et l'option `wholetext` :
chaque fichier devient **une seule ligne**, avec tout son contenu XML dans la colonne `value`.

Sans cette option, Spark créerait une ligne par ligne de texte, ce qui casserait la structure du XML.

Le chemin désigne un **dossier** : tous les fichiers qu'il contient sont lus en une fois.


In [0]:
from pyspark.sql import functions as F

# Dossier du landing contenant les fichiers XML des prix
SOURCE_PATH = "/Volumes/workspace/energy_raw/landing/entsoe/prices/"

# Lecture : une ligne par fichier, tout le contenu XML dans la colonne "value"
df_raw = (
    spark.read
        .option("wholetext", True)
        .text(SOURCE_PATH)
)

df_raw.printSchema()
display(df_raw)

## 2. Métadonnées d'ingestion

Chaque ligne est enrichie de colonnes techniques, préfixées par `_` pour les distinguer des données métier :

| Colonne | Contenu |
|---|---|
| `_source_file` | Nom du fichier XML d'origine |
| `_source_path` | Chemin complet du fichier dans le Volume |
| `_file_modified_at` | Date de dépôt du fichier dans le landing |
| `_ingested_at` | Date de chargement dans la table Bronze |

Ces colonnes assurent la **traçabilité** : pour chaque ligne, on sait d'où elle vient et quand elle a été chargée.

In [0]:
df_bronze = (
    df_raw
        .select(
            # Contenu XML brut, renommé pour être explicite
            F.col("value").alias("xml_content"),
            # Métadonnées fournies par Spark sur le fichier lu
            F.col("_metadata.file_name").alias("_source_file"),
            F.col("_metadata.file_path").alias("_source_path"),
            F.col("_metadata.file_modification_time").alias("_file_modified_at"),
        )
        # Date et heure du chargement
        .withColumn("_ingested_at", F.current_timestamp())
)

df_bronze.printSchema()

# Aperçu sans la colonne XML, trop longue pour être lisible
display(df_bronze.drop("xml_content"))

## 3. Écriture dans la table Delta Bronze

Les données sont écrites dans la table `workspace.energy_bronze.entsoe_prices_raw`, au format Delta.

**Mode `overwrite` :** à chaque exécution, la table est entièrement reconstruite à partir de tous les fichiers du landing.
Le notebook est ainsi **idempotent** : le relancer donne toujours le même résultat, sans doublon.
Cette approche convient tant que le volume reste modeste ; elle sera remplacée plus tard par un chargement incrémental (Auto Loader).


In [0]:
# Table cible au format catalogue.schéma.table
TARGET_TABLE = "workspace.energy_bronze.entsoe_prices_raw"

(
    df_bronze.write
        .format("delta")                     # format Delta Lake
        .mode("overwrite")                   # remplace le contenu à chaque exécution
        .option("overwriteSchema", "true")   # autorise un changement de colonnes
        .saveAsTable(TARGET_TABLE)           # enregistre la table dans Unity Catalog
)

print(f"✔ Table écrite : {TARGET_TABLE}")

In [0]:
df_check = spark.table(TARGET_TABLE)

print(f"Nombre de lignes : {df_check.count()}")
display(df_check.drop("xml_content"))